# KPI-Extraktion aus Geschäftsberichten

Dieses Notebook durchsucht Geschäftsberichte nach zentralen Nachhaltigkeits- und Emissionskennzahlen (KPIs). Die Ergebnisse werden in `data/kpis.csv` gespeichert und bilden die Datengrundlage für die LLM-Auswertung.

## 1. Setup

Benötigte Pakete: `pandas`, `pdfplumber`, `regex`. Optional kann `tabula-py` ergänzt werden, wenn zusätzlich Tabellen extrahiert werden sollen.

In [ ]:
from pathlib import Path
import re
from typing import Dict, List

import pandas as pd
import pdfplumber

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

REPORT_DIR = Path("Annual Reports")
REPORT_PATHS = sorted(REPORT_DIR.glob("*.pdf"))
REPORT_PATHS

## 2. KPI-Regeln

Die Extraktion nutzt reguläre Ausdrücke, um häufige KPI-Formulierungen zu erkennen (z. B. `Scope 1 Emissions`, `Total CO2e`, `Renewable Energy Share`). Bei Bedarf können weitere Muster ergänzt werden.

In [ ]:
METRIC_PATTERNS = [
    ("scope_1_emissions", r"(scope\s*1[^0-9]{0,20})([0-9][0-9.,]*\s*(?:t|tonnes|kt|mt)?\s*(?:co2|co2e))"),
    ("scope_2_emissions", r"(scope\s*2[^0-9]{0,20})([0-9][0-9.,]*\s*(?:t|tonnes|kt|mt)?\s*(?:co2|co2e))"),
    ("scope_3_emissions", r"(scope\s*3[^0-9]{0,20})([0-9][0-9.,]*\s*(?:t|tonnes|kt|mt)?\s*(?:co2|co2e))"),
    ("total_emissions", r"(total[^
]{0,40}emissions[^0-9]{0,20})([0-9][0-9.,]*\s*(?:t|tonnes|kt|mt)?\s*(?:co2|co2e))"),
    ("emission_reduction", r"(reduction[^
]{0,40}emissions[^0-9]{0,20})([0-9][0-9.,]*\s*% )"),
    ("renewable_energy_share", r"(renewable[^
]{0,40}energy[^0-9]{0,20})([0-9][0-9.,]*\s*%)"),
]

def clean_value(value: str) -> str:
    value = value.replace(' ', '')
    value = value.replace(',', '.') if value.count(',') == 1 and value.count('.') == 0 else value
    return value.strip()

def extract_kpis_from_pdf(path: Path) -> List[Dict[str, str]]:
    entries: List[Dict[str, str]] = []
    with pdfplumber.open(path) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            raw_text = page.extract_text() or ""
            text = re.sub(r"\s+", " " , raw_text)
            if not text.strip():
                continue
            lower_text = text.lower()
            for metric, pattern in METRIC_PATTERNS:
                for match in re.finditer(pattern, lower_text, flags=re.IGNORECASE):
                    context_span = match.span()
                    start = max(context_span[0] - 120, 0)
                    end = min(context_span[1] + 120, len(text))
                    context = text[start:end].strip()
                    entries.append({
                        "document_id": path.name,
                        "page": page_number,
                        "metric": metric,
                        "value": clean_value(match.group(2)),
                        "context": context
                    })
    return entries

example_entries = extract_kpis_from_pdf(REPORT_PATHS[0]) if REPORT_PATHS else []
len(example_entries), example_entries[:3]

## 3. Verarbeitung aller Berichte

Für jeden Geschäftsbericht werden die gefundenen KPIs gesammelt und in einem konsolidierten DataFrame abgelegt.

In [ ]:
records: List[Dict[str, str]] = []
for path in REPORT_PATHS:
    records.extend(extract_kpis_from_pdf(path))

kpi_df = pd.DataFrame.from_records(records)
kpi_df.head()

## 4. Export

Die gewonnenen KPIs werden in `data/kpis.csv` gespeichert.

In [ ]:
kpis_csv_path = DATA_DIR / "kpis.csv"
kpi_df.to_csv(kpis_csv_path, index=False)
kpis_csv_path